# LAB-HW-08 — Small FlyBrain Replay / 小网络第一次跑在真实 FPGA

**今天只新增一件事：** 让一个小型 event-driven FlyBrain 教学网络真正运行在 programmable logic（PL）里，再把完整硬件 trace 与同一个 deterministic Python oracle 对账。

前置：LSN-012、LAB-HW-06、LAB-HW-07。

**Project Trace:** RMD-013 · T-HW-008/T-HW-011

这是第一次把前面已经学过的多个概念组合到真实板上；本章不引入 DDR，也不做 performance claim。

## 1. 先把 semantic boundary 说清楚

LAB-HW-08 replay 的是 Lesson 12 原来的 **teaching event machine**。

Lesson 12 已明确说明：它**不是**正式 LIF neuron model；它省略 leak、refractory、最终 fixed-point numerics、concurrent target-write conflict 与正式 valid/ready timing。

所以这一章 PASS 的含义是：

> “已经验证过的 Lesson-12 event-causality 小网络，在 KV260 PL 里执行后得到相同 deterministic trace。”

它**不**表示正式 MOD-004~009 已完成。

## 2. 碰 RTL 之前先冻结完全相同的 network

事实源：

`boards/kv260/fixtures/lab08_four_neuron_replay_v1.json`

fixture 就是 Lesson 12 的四神经元 network：

- 0 → 1，weight +2
- 0 → 2，weight +1
- 1 → 3，weight +2
- 2 → 3，weight +1
- thresholds：`[99, 2, 1, 3]`
- initial accumulator：`[0,0,0,0]`
- initial input queue：`[0]`

expected spike order：`[0,1,2,3]`。

expected final accumulator state：`[0,0,0,0]`。

fixture 完全 deterministic，不使用 PRNG；因此不为了形式完整而伪造一个无意义 seed。

## 3. 让 Python 先定义 expected trace

运行：

```bash
python boards/kv260/runtime/lab08_replay_reference.py \
  --fixture boards/kv260/fixtures/lab08_four_neuron_replay_v1.json \
  --json-out /tmp/lab-hw-08-reference.json
```

必须以 `STATUS=PASS` 结束。

4 个 weighted-event word 冻结为：

```text
0x01020201  # 0 -> 1, +2, accumulator 到 2，spike
0x02010101  # 0 -> 2, +1, accumulator 到 1，spike
0x13020200  # 1 -> 3, +2, accumulator 到 2，不 spike
0x23010301  # 2 -> 3, +1, accumulator 到 3，spike
```

RTL 必须匹配这个 oracle；不能反过来让 RTL 重新定义正确答案。

## 4. 哪些东西在哪里运行？

<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="330" viewBox="0 0 1000 330" role="img" aria-label="LAB-HW-08 host control and replay paths">
  <rect x="25" y="110" width="150" height="90" rx="10" fill="#eef" stroke="#333"/>
  <text x="100" y="143" text-anchor="middle" font-size="14">PS / Linux</text>
  <text x="100" y="168" text-anchor="middle" font-size="12">Python checker</text>
  <rect x="220" y="40" width="180" height="85" rx="10" fill="#efe" stroke="#333"/>
  <text x="310" y="72" text-anchor="middle" font-size="14">AXI BRAM Controller</text>
  <text x="310" y="98" text-anchor="middle" font-size="12">0xA0000000</text>
  <rect x="220" y="210" width="180" height="85" rx="10" fill="#efe" stroke="#333"/>
  <text x="310" y="242" text-anchor="middle" font-size="14">AXI GPIO</text>
  <text x="310" y="268" text-anchor="middle" font-size="12">0xA0010000</text>
  <rect x="455" y="40" width="210" height="85" rx="10" fill="#fee" stroke="#333"/>
  <text x="560" y="72" text-anchor="middle" font-size="14">shared state / trace BRAM</text>
  <text x="560" y="98" text-anchor="middle" font-size="12">4 KiB</text>
  <rect x="455" y="210" width="210" height="85" rx="10" fill="#fee" stroke="#333"/>
  <text x="560" y="242" text-anchor="middle" font-size="14">small replay engine</text>
  <text x="560" y="268" text-anchor="middle" font-size="12">queue → lookup → update</text>
  <rect x="720" y="110" width="240" height="90" rx="10" fill="#fff8dc" stroke="#333"/>
  <text x="840" y="140" text-anchor="middle" font-size="14">spike + weighted-event trace</text>
  <text x="840" y="165" text-anchor="middle" font-size="12">written into reserved BRAM words</text>
  <path d="M175 140 L220 85 M175 175 L220 250 M400 82 L455 82 M400 252 L455 252 M665 250 L720 175 M665 82 L720 140" stroke="#333" stroke-width="2" fill="none"/>
</svg>

重要规则：**`busy=1` 时 host 不访问 BRAM**。这样本章不用同时引入 concurrent BRAM arbitration。

## 5. 把 trace 当作 memory 来读

4 KiB BRAM window 沿用 LAB-HW-07 的 base：`0xA0000000`。

reserved word index：

| words | 含义 |
|---|---|
| 0..3 | final accumulator state |
| 16..19 | spike order |
| 20 | spike count |
| 32..35 | encoded weighted-event trace |
| 36 | event count |

control/status 沿用 LAB-HW-06 的 GPIO base：`0xA0010000`。

status bits：

- bit 0：`busy`
- bit 1：`done`
- bit 2：`error`
- bits 7:4：spike count
- bits 15:8：event count

## 6. build 上板前先证明 RTL replay

```bash
iverilog -g2012 \
  -s kv260_small_replay_engine_tb \
  -o /tmp/lab08_replay \
  boards/kv260/rtl/kv260_replay_state_store.sv \
  boards/kv260/rtl/kv260_small_replay_engine.sv \
  boards/kv260/tb/kv260_small_replay_engine_tb.sv

vvp /tmp/lab08_replay
```

最后应看到：

`PASS: LAB-HW-08 Lesson-12 four-neuron replay trace`

simulation 会检查 final state、spike order、count，以及全部 4 个 encoded weighted event。

## 7. 没有硬件时先跑 differential checker

```bash
python boards/kv260/runtime/small_replay_mmio.py \
  --fixture boards/kv260/fixtures/lab08_four_neuron_replay_v1.json \
  --dry-run \
  --json-out /tmp/lab-hw-08-dry-run.json
```

必须同时出现 `DIFFERENTIAL=PASS` 与 `STATUS=PASS`。

dry-run 证明 checker/oracle behavior，**不**证明 KV260 或 Vivado build。

## 8. Build 并 program LAB-HW-08

development host：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-08-build.log \
  -source boards/kv260/scripts/build_lab08_small_replay.tcl
```

以下情况会阻止 bitstream：

- DRC error；
- 没有 setup/hold timing path；
- setup/hold slack 为负；
- RAMB18/RAMB36 primitive 数量为 0。

expected bitstream：

`build/kv260/lab-hw-08/kv260_small_replay.bit`

保持 Linux 运行；如有 active Kria app 先 unload，然后像 LAB-HW-06/07 一样使用共享 `program_bitstream.tcl` program。

## 9. 执行真实 differential replay

用已经工作的方式把 3 个文件复制到 runtime host：

- `small_replay_mmio.py`
- `lab08_replay_reference.py`
- `lab08_four_neuron_replay_v1.json`

然后：

```bash
sudo python3 /tmp/small_replay_mmio.py \
  --fixture /tmp/lab08_four_neuron_replay_v1.json \
  --json-out /tmp/lab-hw-08-trace.json
```

checker 会：

1. 先计算 Python expected trace；
2. 确认 PL engine idle；
3. 初始化 state 并清掉 stale trace word；
4. 拉起 `start`；
5. `busy=1` 期间只 poll status；
6. 完成后才读取 BRAM trace；
7. 逐字段比较。

实体 PASS 必须同时有 `DIFFERENTIAL=PASS` 和 `STATUS=PASS`。

## 10. Failure class 必须分开

host/replay failure：

- `FIXTURE_ORACLE_INVALID`
- `TRANSPORT_DEVICE_MISSING`
- `TRANSPORT_REQUIRES_ROOT`
- `TRANSPORT_PERMISSION_OR_POLICY`
- `TRANSPORT_MMAP_FAILED`
- `ENGINE_ALREADY_BUSY`
- `ENGINE_REPORTED_ERROR`
- `ENGINE_TIMEOUT`
- `REPLAY_DIFFERENTIAL_MISMATCH`

transport failure 不是 algorithm mismatch；algorithm mismatch 也不是 timing/resource failure。

同样，如果 Ubuntu policy 阻止 `/dev/mem`，不要为了强行 PASS 而降低系统安全设置。

## 11. Expected Evidence / Save Evidence

保留：

- fixture SHA-256；
- `lab08_replay_reference.py` SHA-256；
- `small_replay_mmio.py` SHA-256；
- reference JSON；
- Icarus PASS output；
- Vivado build/timing/utilization/DRC report；
- bitstream SHA-256；
- programming log；
- 完整 physical runtime stdout；
- `lab-hw-08-trace.json`；
- exact expected / observed spike/state/event trace；
- Git commit、Ubuntu/kernel identity、board/carrier revision、date。

cloud-CI PASS 是有用的 engineering evidence，但不是 physical T-HW-008 PASS。

## 12. 分层 debug

replay fail 时按顺序检查：

1. fixture hash 与 Python oracle；
2. open-source RTL simulation；
3. Vivado DRC/timing/resource report；
4. bitstream/programming identity；
5. Linux transport/status；
6. 最后才看 PL differential trace。

这样不会把 stale fixture 或 transport failure 误诊成“神经网络算法错了”。

## 13. Human Check

不看代码解释：

1. 为什么 JSON fixture 是 network 事实源？
2. 为什么先跑 Python replay，再碰 RTL？
3. 为什么 `[0,1,2,3]` 只对这个 Lesson-12 teaching machine 成立？
4. 为什么 `busy=1` 时 host 不访问 BRAM？
5. 为什么不仅比较 final state，还要比较 weighted-event trace？
6. `ENGINE_TIMEOUT` 与 `REPLAY_DIFFERENTIAL_MISMATCH` 有什么区别？
7. 为什么 HW-08 通过仍不代表正式 MOD-004~009 完成？

## 14. Engineering handoff

LAB-HW-08 完成后，第三阶段实体 Lab 完整闭环：

```text
HW-06  PS 能控制 PL
  ↓
HW-07  真实片上 neuron-state memory
  ↓
HW-08  deterministic 小型 event network 在 PL replay
```

下一阶段的问题发生变化：LAB-HW-09 开始把 known data 放到外部 DDR，先证明 integrity，再谈 measurement。